# 06A — Geocoding

**Input:** `Results/05.xlsx`
**Output:** `Results/06A_Final_Geocoding_Catalog.csv`

Resolves substation names to coordinates through the Nominatim API, trying each
country associated with the project before falling back to an unrestricted
search. Regional descriptors, border designations and placeholders are excluded
beforehand, so that the geocoder cannot default to country centroids and create
artificial spatial clustering. Results are labelled by confidence.

Queries depend on the live OpenStreetMap database. The catalogue is shipped so
that later steps reproduce exactly without re-querying.

In [1]:
import pandas as pd
import time
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from geopy.exc import GeocoderTimedOut, GeocoderUnavailable
from tqdm import tqdm

In [2]:
df = pd.read_excel("Results/05.xlsx", header=[0, 1], index_col=0)

In [3]:
# EXCLUSION LIST: Vague, regional, or placeholder entries to be skipped
exclusion_list = [
    # --- Nations & Large Regions (Excluded for spatial accuracy) ---
    'Aragon Region', 'Badenwrtemberg Bavaria', 'Badenwrttemberg Area', 'Belgium', 
    'Bordeaux Area', 'Bretagne', 'Bretagne Sud', 'Danish', 'Dublin Area', 'Dutch', 
    'German', 'Germany', 'Greater Vienna', 'Greater Vienna At', 'Hessebadenwrtemberg', 
    'Lower Saxony', 'Nantes Area', 'North Rhinewestphalia', 'Pamplona Area', 
    'Schleswig Holstein Area', 'Schleswighostein', 'Switzerland', 'Veneto Region',
    'Argyll', 'Hessen-Baden-Württemberg', 'North Rhine-Westphalia', 'Baden-Württemberg Area',

    # --- Cardinal & Generic Areas ---
    'North Finland', 'South Finland', 'Southern Part Of Norway', 'Southern Uk', 
    'Central', 'Northern', 'Southern', 'North Sea', 'Northern Netherlands', 
    'South-East Uk', 'Western Cluster', 'Ohrid Area', 'Region Hamm',

    # --- National Borders ---
    'Austrian National Border', 'Border Area', 'Border Itsi', 'German Border', 
    'Pllt Border', 'Estonian-Latvian Border Area',

    # --- Placeholders & Under Study ---
    'Missing Value', 'To Be Defined', 'Tbc', 'Under Consideration', 
    'Most Appropriate Connection Site Under Study', 'Search Area', 'New', 'Zone', 
    'Splitting Point', 'Tba', 'Not Applicable Connection Point At Seasocket Platform',

    # --- Alternative Options & Multiple Site Lists ---
    'Antwerp Area Or Izegem', 'Marmagne Or Eguzon', 'Dunstown Laois Or Other Tbc', 
    'Woodlands Or New Midlands V Substations Tba', 'Woodlands Or New Midlands V Tba', 
    'Uchtelfangen Or Further', 'Arukla Balti Harku Kiisa Paide Rakvere Sindi Substations', 
    'Gaticahernani Spain Cordemais France And Langageindian Queens', 
    'Great Island And A New In Cork Looping Into The Aghada Knockraha Circuits', 
    'Tba Eg Mongstad Kollsnes Fana', 'Substations In The Baltics', 
    'Is-No-Se-Dk-Grid-Connection', 'New Connection To Lines In The Shannon Estuary',

    # --- Offshore, Platforms & Energy Islands ---
    'Cluster Borwin', 'Cluster Dolwin', 'Cluster Helwin', 'Cluster Sylwin', 'Onshore',
    'Danish Energy Island', 'Danish Island', 'On Danish Energy Island', 
    'Elwind Offshore', 'North Atlantic Oss', 'Owf Cluster Baltic Sea', 
    'Owf Cluster Baltic Sea East', 'Platform', 'Princess Elisabeth Island', 
    'Two New Platforms Located Within The Dmap Area', 'Western Saaremaa Offshore', 
    'Wind Farm Cluster Baltic Sea West', 'Bornholm Energy Island', 'Oss Tramontane', 
    'Oss Mistral', 'Platform With Equipment For Reactive Power Compensation',

    # --- Generic Infrastructure & Descriptive Sites ---
    'Athens Site', 'Hadera Site', 'Kofinou Site', 'Korakia Site', 'Vasilikos Site', 
    'Converter In Malta', 'In Italy It Ragusa Terminal', 'In Malta Mt Maghtab Terminal', 
    'German Grid Connection', 'German Located In The Area North Of Bremen', 
    'German Located In The Rastede', 'New In South Donegal', 'New On Gotland', 
    'East Anglia Connection Node', 'Srvest F Windfarm', 'Wind Farm', 
    'Wind Park Nordergrnde', 'Wind Park Riffgat', 'Swiss Node', 'Se','Ree', 'Svenska Kraftnät', 
    'Terna', 'Rte'
]

In [4]:
# Updated Correction Map with all new manual verification
correction_map = {
    # --- ITALY (Sud-Nord & Omonimia) ---
    'Sorgente': 'Sorgente, Messina, Italy',
    'Priolo': 'Priolo Gargallo, Sicily, Italy',
    'Laino': 'Laino Borgo, Calabria, Italy',
    'Melilli': 'Melilli, Sicily, Italy',
    'Rossano': 'Rossano Calabro, Italy',
    'Scilla': 'Scilla, Calabria, Italy',
    'Rizziconi': 'Rizziconi, Calabria, Italy',

    # --- SPAIN (Correzioni Regionali) ---
    'Mezquita': 'Mezquita de Jarque, Teruel, Spain',     # Mezquita-Morella
    'Gatica': 'Gatica, Biscay, Spain',                  # Gatica-Guenes
    'D.Rodrigo': 'Don Rodrigo, Seville, Spain',         # D.Rodrigo-Aljarafe
    'Arcos': 'Arcos de la Frontera, Cadiz, Spain',      # Cartuja-Arcos

    # --- UK & NORDICS ---
    'Sima': 'Sima, Eidfjord, Norway',                   # Norway-UK Link
    'Hunterston': 'Hunterston, North Ayrshire, Scotland',
    'Deeside': 'Deeside, Flintshire, Wales',

    # --- BALCANS & EST EUROPE ---
    'Visegrad': 'Višegrad, Bosnia and Herzegovina',
    'Arad': 'Arad, Romania',                            # West Romania
    'Filippi': 'Filippoi, Kavala, Greece',              # North Greece
    'N.Santa': 'Nea Santa, Kilkis, Greece',             # Nea Santa (Greece)
    'Maritsa': 'Maritsa Iztok, Bulgaria',
    'Khae': 'Kruonis Hydroelectric, Lithuania',
    'Bitenai': 'Bitenai, Lithuania',
    'Alytus': 'Alytus, Lithuania',

    # --- CENTRAL EUROPE ---
    'Muhlbach': 'Mühlbach, Baden-Württemberg, Germany', # DE-FR Border

    # --- PORTUGAL ---
    'V.Minho': 'Vila do Conde, Portugal',
    'A.V. Alentejo': 'Ferreira do Alentejo, Portugal',
    
    # --- ROMANIA & BULGARIA ---
    'Smardan': 'Smardan, Galați, Romania',               
    
    # --- FRANCE ---
    'Mandarins': 'Mandarins, Coquelles, France',         
    'Grande Ile': 'Grande Ile, Savoie, France',          
    'Marmagne': 'Marmagne, Cher, France',                
    
    # --- POLAND ---
    'Baczyna': 'Baczyna, Lubusz, Poland',                
    'Zarnowiec': 'Żarnowiec, Pomeranian Voivodeship, Poland', 
    
    # --- GREAT BRITAIN ---
    'Seabank': 'Seabank, Avonmouth, Bristol, UK',        
    'Coleraine': 'Coleraine, Northern Ireland, UK',      
    'Hinkley Point': 'Hinkley Point C, Somerset, UK',    
    
    # --- OFFSHORE (German North Sea) ---
    'NOR-13': 'German Bight, North Sea',                 
    'NOR-13-1': 'German Bight, North Sea',
    'NOR-13-2': 'German Bight, North Sea',

    # --- ENCODING ERRORS & TYPOS ---
    'Siedlce UjrzanÃ³w': 'Siedlce Ujrzanow, Poland', # Fixes encoding and country mismatch (was mislabeled as BEL)
    'Bescan': 'Bescanó, Girona, Spain',              # Full name with accent and province
    'Bericevo': 'Bericevo, Slovenia',                # Fixes "Berievo" typo common in databases
    'Fillipi': 'Philippoi, Kavala, Greece',          # Correct Greek name for the northern node
    'Giai': 'Gižai, Lithuania',                      # Critical Baltic synchronization node
    'Pila Krzewina': 'Piła Krzewina, Poland',        # Polish special characters
    'Le Mandarins': 'Mandarins, Coquelles, France',  # ElecLink landing point
    'Artsyz': 'Artsyz, Odesa Oblast, Ukraine',       # Interconnection node with Romania
    
    # --- COUNTRY MISMATCHES ---
    'Neuravensburg': 'Neuravensburg, Germany',       # Moved from AUT (Austria) to Germany
    'Dunstown': 'Dunstown, County Kildare, Ireland', # Key UK-IE interconnection node
    'Ni Kilroot': 'Kilroot Power Station, Northern Ireland',
    'Norwich Main': 'Norwich Main Substation, UK',
    'Balti': 'Balti Power Plant, Narva, Estonia',
    
    # --- BELGIUM (Ventilus / Princess Elisabeth Island) ---
    'Onshore': 'Stevin Substation, Zeebrugge, Belgium', # Actual onshore landing point for the offshore link
    
    # --- EGYPT (Landing point "Nile West" -> Burullus) ---
    'Nile West Point': 'New Burullus Power Plant, Egypt', # Real coastal hub for subsea cables
    'Nile West': 'New Burullus Power Plant, Egypt',       
    'Mersa Matruh': 'Marsa Matruh, Egypt',
    'Damietta': 'Damietta, Egypt',
    
    # --- OFFSHORE & MEDITERRANEAN PROJECTS ---
    'Anaklia': 'Anaklia, Georgia',                   # Black Sea Cable node
    'Kofinou': 'Kofinou, Cyprus',                    # EuroAsia Interconnector node
    'Tobruk': 'Tobruk, Libya',                       # Landing point for the link with Greece
    
    # --- TRUNCATED OR GENERIC NAMES ---
    'Olron': 'Oléron Island, France',                # Reference to the offshore wind farm
    'Burgenland North': 'Neusiedl am See, Austria',  # Center of the Austrian wind cluster
    'Panias': 'Panias, Portugal',                    
    'Frido': 'Frido, Portugal',                      
    'Grobina': 'Grobiņa, Latvia',
    'Imanta': 'Imanta, Riga, Latvia',
    'Telsiai': 'Telšiai, Lithuania'
}

In [5]:
# Map ISO3 codes to English country names for better geocoding results
iso3_full_map = {
    'FRA': 'France', 'ESP': 'Spain', 'ITA': 'Italy', 'DEU': 'Germany', 
    'PRT': 'Portugal', 'GBR': 'United Kingdom', 'BEL': 'Belgium', 
    'NLD': 'Netherlands', 'CHE': 'Switzerland', 'POL': 'Poland',
    'AUT': 'Austria', 'CZE': 'Czechia', 'SVN': 'Slovenia', 'GRC': 'Greece',
    'SRB': 'Serbia', 'MNE': 'Montenegro', 'BIH': 'Bosnia and Herzegovina',
    'ALB': 'Albania', 'MKD': 'North Macedonia', 'BGR': 'Bulgaria',
    'ROU': 'Romania', 'HUN': 'Hungary', 'SVK': 'Slovakia', 'HRV': 'Croatia',
    'FIN': 'Finland', 'SWE': 'Sweden', 'NOR': 'Norway', 'DNK': 'Denmark',
    'EST': 'Estonia', 'LVA': 'Latvia', 'LTU': 'Lithuania', 'IRL': 'Ireland',
    'TUN': 'Tunisia', 'DZA': 'Algeria', 'MAR': 'Morocco', 'TUR': 'Turkey',
    'AZE': 'Azerbaijan', 'CYP': 'Cyprus','EGY': 'Egypt', 'GEO': 'Georgia',
    'ISL': 'Iceland', 'ISR': 'Israel', 'ISREAL': 'Israel', 'LBY': 'Libya', 
    'LUX': 'Luxembourg', 'MDA': 'Moldova', 'MLT': 'Malta', 'UKR': 'Ukraine'
}

In [6]:
df[('meta', 'Inv_Substation_From_Final')] = df[('meta', 'Inv_Substation_From_Final')].replace(correction_map)
df[('meta', 'Inv_Substation_To_Final')] = df[('meta', 'Inv_Substation_To_Final')].replace(correction_map)

from_subs = df[[('meta', 'Inv_Substation_From_Final'), ('meta', 'Project_Country_ISO3')]].copy()
from_subs.columns = ['Name', 'Country_ISO3']

to_subs = df[[('meta', 'Inv_Substation_To_Final'), ('meta', 'Project_Country_ISO3')]].copy()
to_subs.columns = ['Name', 'Country_ISO3']

catalog = pd.concat([from_subs, to_subs], ignore_index=True)
catalog = catalog.dropna(subset=['Name'])

catalog = catalog.drop_duplicates(subset=['Name', 'Country_ISO3'])

catalog = catalog[~catalog['Name'].isin(exclusion_list)]

🏗️ Extracting unique substations from meta columns...
📍 Catalog ready: 944 unique items to geocode.


In [7]:

geolocator = Nominatim(user_agent="eu_grid_geocoder_v5", timeout=10)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.1, max_retries=3)

In [8]:

def find_location_smart(row):
    """
    Tries to find the substation in each country listed for the project.
    Includes anti-duplicate logic for country names.
    """
    try:
        sub_name = str(row['Name'])
        iso_list = str(row['Country_ISO3']).split(';')
        
        for iso in iso_list:
            country_full = iso3_full_map.get(iso.strip(), "")
            if country_full:
                if country_full.lower() in sub_name.lower():
                    query = sub_name
                else:
                    query = f"{sub_name}, {country_full}"
                
                try:
                    location = geocode(query)
                    if location:
                        return pd.Series([location.latitude, location.longitude, f"Found in {iso}"])
                except (GeocoderTimedOut, GeocoderUnavailable):
                    time.sleep(2)

        try:
            location = geocode(sub_name)
            if location:
                return pd.Series([location.latitude, location.longitude, "Found (Global Search)"])
        except:
            pass

    except Exception:
        return pd.Series([None, None, "Error"])

    return pd.Series([None, None, "Not Found"])

In [9]:
tqdm.pandas(desc="Geocoding Substations")

catalog[['Lat', 'Lon', 'Status']] = catalog.progress_apply(find_location_smart, axis=1)

found = catalog[catalog['Status'].str.contains("Found")]
print("-" * 30)
print(f"Success: {len(found)} / {len(catalog)} ({len(found)/len(catalog):.1%})")

⏳ Starting smart geocoding (attempting all project countries)...


Geocoding Substations: 100%|██████████| 944/944 [25:31<00:00,  1.62s/it]

------------------------------
✅ Success: 944 / 944 (100.0%)


In [10]:
catalog.to_csv("Results/06A_Final_Geocoding_Catalog.csv", index=False, sep=';')